# 1 Загрузка данных

In [1]:
import pandas as pd
import numpy as np

data = {
    "order_id": [1001,1002,1003,1004,1005,1006,1007,1008],
    "customer_id": ["C-101","C-102","C-101","C-103","C-104","C-102","C-105","C-103"],
    "order_date": ["2024-03-01","2024-03-01","2024/03/02","2024-03-03","03.03.2024","2024-03-04","2024-03-05","2024-03-05"],
    "product": ["Laptop", np.nan, "Laptop", "Mouse", "Keyboard", "Monitor", "laptop", "Mouse"],
    "quantity": [1,2,-1,3,1,1,2,0],
    "price": [75000,3500,75000,800,np.nan,22000,75000,800],
    "city": ["Москва","спб","Москва","МОСКВА","Казань","СПб"," ","москва"]
}

df = pd.DataFrame(data)

# 2 Исследование данных

In [2]:
print("\nОБЩАЯ ИНФОРМАЦИЯ")
print(df.info())

print("\nВСЕГО СТРОК")

rows_before = len(df)
print(rows_before)

print("\nПРОПУСКИ")
print(df.isna().sum())


ОБЩАЯ ИНФОРМАЦИЯ
<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     8 non-null      int64  
 1   customer_id  8 non-null      str    
 2   order_date   8 non-null      str    
 3   product      7 non-null      str    
 4   quantity     8 non-null      int64  
 5   price        7 non-null      float64
 6   city         8 non-null      str    
dtypes: float64(1), int64(2), str(4)
memory usage: 580.0 bytes
None

ВСЕГО СТРОК
8

ПРОПУСКИ
order_id       0
customer_id    0
order_date     0
product        1
quantity       0
price          1
city           0
dtype: int64


In [3]:
print("\nПОИСК АНОМАЛИЙ")
print(df.describe())

print("\nКОЛИЧЕСТВО УНИКАЛЬНЫХ ЗНАЧЕНИЙ")
print(df.nunique())


ПОИСК АНОМАЛИЙ
         order_id  quantity         price
count     8.00000  8.000000      7.000000
mean   1004.50000  1.125000  36014.285714
std       2.44949  1.246423  37178.238643
min    1001.00000 -1.000000    800.000000
25%    1002.75000  0.750000   2150.000000
50%    1004.50000  1.000000  22000.000000
75%    1006.25000  2.000000  75000.000000
max    1008.00000  3.000000  75000.000000

КОЛИЧЕСТВО УНИКАЛЬНЫХ ЗНАЧЕНИЙ
order_id       8
customer_id    5
order_date     6
product        5
quantity       5
price          4
city           7
dtype: int64


# 3 Очистка данных

In [4]:
start_count = len(df)

df = df.replace(r'^\s*$', np.nan, regex=True, inplace=False)
df['city'] = df['city'].fillna('Unknown')

df = df[df['quantity'] > 0]
print(f"\nУдалено строк из-за некорректного количества (<= 0): {start_count - len(df)}")

df['city'] = df['city'].str.lower().str.strip()

city_correction = {
    'спб': 'Санкт-Петербург',
    'москва': 'Москва',
    'казань': 'Казань',
    'unknown': 'Unknown'
}

df['city'] = df['city'].replace(city_correction)

df['product']=df['product'].str.strip().str.lower()


df['order_date'] = pd.to_datetime(df['order_date'], format='mixed', dayfirst=False)

print("\nАНАЛИЗ ВОЗМОЖНОСТИ ВОССТАНОВЛЕНИЯ:")
# 1. Проверяем клавиатуру (ID 1005)
keyboard_check = df[df['product'] == 'keyboard']
print(f"Строки с Keyboard:\n{keyboard_check}") 
# "Для товара 'keyboard' (ID 1005) цена отсутствует. В датасете нет других клавиатур, 
# чтобы восстановить цену через среднее или медиану. Строка подлежит удалению."

# 2. Проверяем пропуск в product (ID 1002)
missing_product = df[df['price'] == 3500]
print(f"\nСтроки с ценой 3500:\n{missing_product}")
# "Для ID 1002 название товара отсутствует. Цена 3500 уникальна и не встречается 
# у других товаров, восстановить название по цене невозможно."

# --- ТЕПЕРЬ УДАЛЯЕМ ---
before_dropna = len(df)
df = df.dropna(subset=['product', 'price'])
print(f"\nУдалено из-за пустых цен/названий: {before_dropna - len(df)}")

df = df.reset_index(drop=True)



Удалено строк из-за некорректного количества (<= 0): 2

АНАЛИЗ ВОЗМОЖНОСТИ ВОССТАНОВЛЕНИЯ:
Строки с Keyboard:
   order_id customer_id order_date   product  quantity  price    city
4      1005       C-104 2024-03-03  keyboard         1    NaN  Казань

Строки с ценой 3500:
   order_id customer_id order_date product  quantity   price             city
1      1002       C-102 2024-03-01     NaN         2  3500.0  Санкт-Петербург

Удалено из-за пустых цен/названий: 2


# 4 Расчеты и Выводы

In [5]:
df['total'] = df['price'] * df['quantity']

rows_after = len(df)
deleted_count = rows_before - rows_after

print(f"ОБЩАЯ ВЫРУЧКА: {df['total'].sum():,.2f} руб.")
print(f"Удалено мусорных строк: {deleted_count}")
print(f"Осталось чистых строк: {rows_after}")

print("\nИТОГОВЫЙ ДАТАФРЕЙМ")
display(df)

print("\nФИНАЛЬНАЯ ПРОВЕРКА ЧИСТОТЫ:")
print(f"Осталось NaN в таблице: {df.isna().sum().sum()}")
print(f"Минимальное количество в заказе: {df['quantity'].min()}")

ОБЩАЯ ВЫРУЧКА: 249,400.00 руб.
Удалено мусорных строк: 4
Осталось чистых строк: 4

ИТОГОВЫЙ ДАТАФРЕЙМ


,order_id,customer_id,order_date,product,quantity,price,city,total
0,1001,C-101,2024-03-01,laptop,1,75000.0,Москва,75000.0
1,1004,C-103,2024-03-03,mouse,3,800.0,Москва,2400.0
2,1006,C-102,2024-03-04,monitor,1,22000.0,Санкт-Петербург,22000.0
3,1007,C-105,2024-03-05,laptop,2,75000.0,Unknown,150000.0



ФИНАЛЬНАЯ ПРОВЕРКА ЧИСТОТЫ:
Осталось NaN в таблице: 0
Минимальное количество в заказе: 1


In [8]:
# Вместо двойного вызова groupby
city_revenue = df.groupby('city')['total'].sum()

top_city_name = city_revenue.idxmax()
top_city_value = city_revenue.max()

print(f"Город-лидер: {top_city_name} (Выручка: {top_city_value:,.2f})")

client_counts = df.groupby('customer_id')['order_id'].count()
max_orders = client_counts.max()
top_clients = client_counts[client_counts == max_orders]

print(f"Клиент который больше заказывает: {top_clients}")

avg_cost = df.groupby('product')['total'].mean().sort_values(ascending=False)
print(f"\nСредний чек по товарам:\n{avg_cost}")

Город-лидер: Unknown (Выручка: 150,000.00)
Клиент который больше заказывает: customer_id
C-101    1
C-102    1
C-103    1
C-105    1
Name: order_id, dtype: int64

Средний чек по товарам:
product
laptop     112500.0
monitor     22000.0
mouse        2400.0
Name: total, dtype: float64
